In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

## Introduction & Methodology
This notebook conducts a systematic benchmark of five supervised classification algorithms on the **Breast Cancer Wisconsin (Diagnostic)** dataset (`569` samples, `30` continuous features) to identify the most reliable architecture for detecting malignant tumors.

* **Target Formulation:** Labels are inverted from the default `scikit-learn` encoding so that **Malignant = 1 (Positive Class)** and **Benign = 0 (Negative Class)**, aligning evaluation metrics (specifically **Recall** and **F1-score**) with clinical diagnostic priorities.
* **Data Partitioning:** The dataset is split using a stratified `80/20` ratio (`455` training samples, `114` test samples, `random_state=1234`) to preserve class proportions across subsets.
* **Leak-Free Preprocessing:** Distance- and margin-based algorithms (**Logistic Regression**, **KNN**, and **RBF SVM**) are wrapped inside `scikit-learn` `Pipeline` objects with `StandardScaler`, ensuring feature scaling is fitted independently within each cross-validation fold. Tree-based models (**Decision Tree** and **Random Forest**) are trained on unscaled features, as split thresholds are invariant to monotonic scaling.

In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns = data.feature_names)
y = pd.Series(data.target, name='target')

# Set malignant as the positive class
y = pd.Series((data.target == 0).astype(int), name='target')

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= 0.2,
    random_state= 1234,
    stratify= y
)

## Stratified 5-Fold Cross-Validation Results (`n=455`)
Models are evaluated on the training set using `StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)` across Accuracy, Precision, Recall, and F1-score, alongside mean training accuracy to diagnose overfitting:

| Model | Configuration | CV Accuracy (Mean ± Std) | CV Precision (Mean ± Std) | CV Recall (Mean ± Std) | CV F1-Score (Mean ± Std) | Train Accuracy |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **SVM (RBF)** | `C=1, gamma='scale'` | **`0.9780 ± 0.0120`** | **`0.9822 ± 0.0146`** | **`0.9588 ± 0.0300`** | **`0.9700 ± 0.0166`** | `0.9841` |
| **Logistic Regression** | `max_iter=5000` | `0.9758 ± 0.0189` | `0.9821 ± 0.0233` | `0.9529 ± 0.0399` | `0.9669 ± 0.0263` | `0.9896` |
| **KNN** | `n_neighbors=5` | `0.9648 ± 0.0176` | `0.9820 ± 0.0148` | `0.9235 ± 0.0546` | `0.9507 ± 0.0256` | `0.9703` |
| **Random Forest** | `n_estimators=200` | `0.9538 ± 0.0213` | `0.9529 ± 0.0400` | `0.9235 ± 0.0353` | `0.9374 ± 0.0288` | `1.0000` |
| **Decision Tree** | Default (`CART`) | `0.9341 ± 0.0184` | `0.9178 ± 0.0315` | `0.9059 ± 0.0432` | `0.9110 ± 0.0264` | `1.0000` |

### Cross-Validation Insights
* **Top Performers & Stability:** **RBF SVM** achieves the highest mean validation scores across all four metrics while exhibiting the lowest fold-to-fold variance (`Accuracy Std = 0.0120`, `F1 Std = 0.0166`), closely followed by **Logistic Regression**.
* **Overfitting Diagnosis:** Both **Decision Tree** and **Random Forest** reach `1.0000` training accuracy (`0%` training error), indicating high variance and overfitting to the training folds compared to the regularized linear and kernel models.

In [3]:
# 5-Fold Stratified Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1234
)

# Define models
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000))
    ]),

    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),

    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1, gamma='scale'))
    ]),

    'Decision Tree': DecisionTreeClassifier(
        random_state=1234
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=1234
    )
}

# Evaluation metrics
scoring = ['accuracy', 'precision', 'recall', 'f1']

results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True
    )

    results.append({
        'Model': name,

        'Accuracy Mean': scores['test_accuracy'].mean(),
        'Accuracy Std': scores['test_accuracy'].std(),

        'Precision Mean': scores['test_precision'].mean(),
        'Precision Std': scores['test_precision'].std(),

        'Recall Mean': scores['test_recall'].mean(),
        'Recall Std': scores['test_recall'].std(),

        'F1 Mean': scores['test_f1'].mean(),
        'F1 Std': scores['test_f1'].std(),

        'Train Accuracy': scores['train_accuracy'].mean()
    })

results_df = pd.DataFrame(results)



In [4]:
print(results_df.round(4).to_string(index=False))

              Model  Accuracy Mean  Accuracy Std  Precision Mean  Precision Std  Recall Mean  Recall Std  F1 Mean  F1 Std  Train Accuracy
Logistic Regression         0.9758        0.0189          0.9821         0.0233       0.9529      0.0399   0.9669  0.0263          0.9896
                KNN         0.9648        0.0176          0.9820         0.0148       0.9235      0.0546   0.9507  0.0256          0.9703
                SVM         0.9780        0.0120          0.9822         0.0146       0.9588      0.0300   0.9700  0.0166          0.9841
      Decision Tree         0.9341        0.0184          0.9178         0.0315       0.9059      0.0432   0.9110  0.0264          1.0000
      Random Forest         0.9538        0.0213          0.9529         0.0400       0.9235      0.0353   0.9374  0.0288          1.0000


## Holdout Test Set Evaluation (`n=114`)
Each model is refitted on the full training set (`455` samples) and evaluated on the unseen holdout test set (`72` Benign, `42` Malignant):

| Model | Test Accuracy | Test Precision | Test Recall | Test F1-Score |
| :--- | :---: | :---: | :---: | :---: |
| **KNN** | **`0.9825`** | **`0.9762`** | **`0.9762`** | **`0.9762`** |
| **SVM (RBF)** | `0.9737` | `0.9535` | **`0.9762`** | `0.9647` |
| **Random Forest** | `0.9737` | `0.9535` | **`0.9762`** | `0.9647` |
| **Logistic Regression** | `0.9561` | `0.9302` | `0.9524` | `0.9412` |
| **Decision Tree** | `0.9211` | `0.8667` | `0.9286` | `0.8966` |

In [4]:
final_models = {
    'Logistic Regression': models['Logistic Regression'],
    'KNN': models['KNN'],
    'SVM': models['SVM'],
    'Decision Tree': models['Decision Tree'],
    'Random Forest': models['Random Forest']
}

test_results = []

for name, model in final_models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    test_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    })

test_results_df = pd.DataFrame(test_results)

print(test_results_df.round(4).to_string(index=False))

              Model  Accuracy  Precision  Recall     F1
Logistic Regression    0.9561     0.9302  0.9524 0.9412
                KNN    0.9825     0.9762  0.9762 0.9762
                SVM    0.9737     0.9535  0.9762 0.9647
      Decision Tree    0.9211     0.8667  0.9286 0.8966
      Random Forest    0.9737     0.9535  0.9762 0.9647


## Conclusion & Model Selection
* **Best Overall Generalization (`SVM`):** Although **KNN** edges out the other models on the single 114-sample holdout split (`Accuracy = 0.9825`, `F1 = 0.9762`), **RBF SVM** demonstrates superior stability and generalization across Stratified 5-Fold Cross-Validation (`CV F1 = 0.9700 ± 0.0166` vs. KNN's `0.9507 ± 0.0256`, where KNN recall fluctuates with `Std = 0.0546`).
* **Clinical Sensitivity:** **SVM**, **KNN**, and **Random Forest** all tie for the highest test sensitivity (`Recall = 0.9762`), correctly identifying `41` out of `42` malignant tumors in the test set.
* **Tree vs. Margin/Distance Models:** Unpruned **Decision Trees** suffer from substantial variance (`Test Accuracy = 0.9211`), and while **Random Forest** ensembling (`200` trees) recovers `+5.26%` in test accuracy, continuous margin-based (**SVM**) and linear (**Logistic Regression**) models remain better suited for standardized, continuous morphological features.